In [1]:
import os
os.chdir("..")

In [2]:
import numpy as np
import pandas as pd

def information_value(x: pd.Series, y: pd.Series, bins: int = 10) -> float:
    if pd.api.types.is_numeric_dtype(x) and x.nunique() > bins:
        groups = pd.qcut(x, q=bins, duplicates="drop")  # numeric: decile bins
    else:
        groups = x  # categorical or low-cardinality: one bin per value
    groups = groups.astype("object").where(x.notna(), "Missing")  # missing as its own bin
    table = pd.crosstab(groups, y)
    good = (table[0] + 0.5) / table[0].sum()  # +0.5 avoids log(0) in empty bins
    bad = (table[1] + 0.5) / table[1].sum()
    woe = np.log(good / bad)
    return float(((good - bad) * woe).sum())

In [3]:
from sklearn.model_selection import train_test_split

df = pd.read_csv("data/raw/application_train.csv")
df = df[df["CODE_GENDER"] != "XNA"]
df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].mask(df["DAYS_EMPLOYED"] == 365243)

train, test = train_test_split(df, test_size=0.2, stratify=df["TARGET"], random_state=42)

In [4]:
candidates = [c for c in df.columns if c not in ("SK_ID_CURR", "TARGET")]
iv = pd.Series(
    {c: information_value(train[c], train["TARGET"]) for c in candidates}
).sort_values(ascending=False)
iv.head(25)

EXT_SOURCE_3                   0.328043
EXT_SOURCE_2                   0.303499
EXT_SOURCE_1                   0.152557
DAYS_EMPLOYED                  0.112545
AMT_GOODS_PRICE                0.089696
DAYS_BIRTH                     0.087158
OCCUPATION_TYPE                0.083693
ORGANIZATION_TYPE              0.074368
NAME_INCOME_TYPE               0.056747
REGION_RATING_CLIENT_W_CITY    0.051670
NAME_EDUCATION_TYPE            0.049468
REGION_RATING_CLIENT           0.048295
DAYS_LAST_PHONE_CHANGE         0.044239
AMT_CREDIT                     0.043946
CODE_GENDER                    0.038489
FLOORSMAX_AVG                  0.037759
FLOORSMAX_MEDI                 0.037665
FLOORSMAX_MODE                 0.037168
DAYS_ID_PUBLISH                0.036412
TOTALAREA_MODE                 0.035216
REGION_POPULATION_RELATIVE     0.034744
FLAG_EMP_PHONE                 0.033948
REG_CITY_NOT_WORK_CITY         0.033653
LIVINGAREA_AVG                 0.032891
LIVINGAREA_MEDI                0.032488


In [8]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

def fit_auc(features):
    X_train, X_test = train[features].copy(), test[features].copy()
    for col in features:
        if not pd.api.types.is_numeric_dtype(X_train[col]):
            X_train[col] = X_train[col].astype("category")
            X_test[col] = pd.Categorical(X_test[col], categories=X_train[col].cat.categories)
    model = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, random_state=42, verbose=-1)
    model.fit(X_train, train["TARGET"])
    return roc_auc_score(test["TARGET"], model.predict_proba(X_test)[:, 1])

In [6]:
EXCLUDED = {
    "CODE_GENDER": "sex: prohibited in EU credit (Directive 2004/113/EC)",
    "NAME_FAMILY_STATUS": "family status: sensitive, excluded as a precaution",
    "CNT_CHILDREN": "family status proxy",
    "CNT_FAM_MEMBERS": "family status proxy",
}

eligible = iv.drop(list(EXCLUDED), errors="ignore")
ranked = [f for f in eligible.index if f != "REGION_RATING_CLIENT"]  # near-duplicate of _W_CITY
candidate = ranked[:11] + ["AMT_CREDIT"]  # keep AMT_CREDIT for its interaction with AMT_GOODS_PRICE
candidate

['EXT_SOURCE_3',
 'EXT_SOURCE_2',
 'EXT_SOURCE_1',
 'DAYS_EMPLOYED',
 'AMT_GOODS_PRICE',
 'DAYS_BIRTH',
 'OCCUPATION_TYPE',
 'ORGANIZATION_TYPE',
 'NAME_INCOME_TYPE',
 'REGION_RATING_CLIENT_W_CITY',
 'NAME_EDUCATION_TYPE',
 'AMT_CREDIT']

In [9]:
from credit_risk_api.train import FEATURES as current

iv_top12 = iv.head(12).index.tolist()
adjusted = [f for f in iv_top12 if f != "REGION_RATING_CLIENT"] + ["AMT_CREDIT"]
no_org = [f for f in adjusted if f != "ORGANIZATION_TYPE"]

for name, feats in [("current", current), ("iv_top12", iv_top12),
                    ("adjusted", adjusted), ("adjusted_no_org", no_org)]:
    print(f"{name:16} {len(feats)} features  AUC = {fit_auc(feats):.4f}")

current          12 features  AUC = 0.7538
iv_top12         12 features  AUC = 0.7470
adjusted         12 features  AUC = 0.7493
adjusted_no_org  11 features  AUC = 0.7501


In [11]:
SENSITIVE = {
    "CODE_GENDER", "DAYS_BIRTH",                               # sex, age
    "NAME_FAMILY_STATUS", "CNT_CHILDREN", "CNT_FAM_MEMBERS",   # family status
    "REGION_RATING_CLIENT", "REGION_RATING_CLIENT_W_CITY",     # geography
    "REGION_POPULATION_RELATIVE",
}
STRICT = SENSITIVE | {"NAME_EDUCATION_TYPE"}

current_safe = [f for f in current if f not in SENSITIVE]
current_strict = [f for f in current if f not in STRICT]

variants = {
    "current (reference)": current,
    "current_safe": current_safe,
    "current_strict": current_strict,
    "current_safe + income_type": current_safe + ["NAME_INCOME_TYPE"],
    "current_safe + occupation": current_safe + ["OCCUPATION_TYPE"],
}
for name, feats in variants.items():
    print(f"{name:28} {len(feats)} features  AUC = {fit_auc(feats):.4f}")

current (reference)          12 features  AUC = 0.7538
current_safe                 10 features  AUC = 0.7492
current_strict               9 features  AUC = 0.7461
current_safe + income_type   11 features  AUC = 0.7490
current_safe + occupation    11 features  AUC = 0.7519
